In [1]:
# Import necessary libraries
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from joblib import Parallel, delayed

# Load the dataset
# Note: Ensure that "class1_dataset.xlsx" is in your working directory or provide the correct path.
data = pd.read_excel("class123_dataset.xlsx")

# Extract the predictors and outcome
X = data.drop('RRI', axis=1)
Y = data['RRI']

# List of alpha values to try (for feature selection)
alpha_values = np.logspace(10, -1, 20)

# Define a single Logistic Regression classifier with default hyperparameters
classifiers = [
    ("Logistic Default", LogisticRegression(random_state=42, max_iter=1000))
]

# Dictionary to store the best AUC, feature combination, and alpha for each classifier
best_results_by_classifier = {name: (0, None, None) for name, _ in classifiers}

def compute_best_result(alpha, classifier_name, model, X, Y):
    """
    Compute the best ROC AUC score for a given alpha and classifier.

    Parameters:
    - alpha: Regularization strength for feature selection (inverse of C).
    - classifier_name: Name of the classifier.
    - model: The classifier instance.
    - X: All features.
    - Y: All labels.

    Returns:
    - Tuple containing classifier name, best AUC, best feature indices, and alpha.
    """
    # Initialize L1-penalized Logistic Regression for feature selection
    lasso = LogisticRegression(penalty='l1', C=1/alpha, solver='saga', max_iter=10000, random_state=42)

    # Fit the Lasso model on the entire dataset
    lasso.fit(X, Y)

    # Get the coefficients (they are returned as a 2D array, so flatten it)
    coefficients = lasso.coef_[0]

    # Extract non-zero coefficients and sort them by absolute value
    non_zero_coefficients = sorted(
        [(index, coef) for index, coef in enumerate(coefficients) if coef != 0],
        key=lambda x: abs(x[1]),
        reverse=True
    )

    max_auc_for_alpha = 0
    best_feature_combination_for_alpha = None

    for i in range(1, len(non_zero_coefficients) + 1):
        top_features_indices = [index for index, _ in non_zero_coefficients[:i]]
        X_reduced = X.iloc[:, top_features_indices]
        skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
        aucs = []
        for train_index, test_index in skf.split(X_reduced, Y):
            X_train_fold, X_test_fold = X_reduced.iloc[train_index], X_reduced.iloc[test_index]
            y_train_fold, y_test_fold = Y.iloc[train_index], Y.iloc[test_index]
            model.fit(X_train_fold, y_train_fold)
            if hasattr(model, "predict_proba"):
                y_pred_probs_fold = model.predict_proba(X_test_fold)[:, 1]
            else:
                # For classifiers that do not have predict_proba, use decision_function
                y_pred_probs_fold = model.decision_function(X_test_fold)
                # Scale the decision function to [0,1] using min-max scaling
                y_pred_probs_fold = (y_pred_probs_fold - y_pred_probs_fold.min()) / (y_pred_probs_fold.max() - y_pred_probs_fold.min() + 1e-8)
            aucs.append(roc_auc_score(y_test_fold, y_pred_probs_fold))

        mean_auc = np.mean(aucs)
        if mean_auc > max_auc_for_alpha:
            max_auc_for_alpha = mean_auc
            best_feature_combination_for_alpha = top_features_indices

    return (classifier_name, max_auc_for_alpha, best_feature_combination_for_alpha, alpha)

# Create a list of all (alpha, classifier) pairs with data
tasks = [(alpha, name, model, X, Y) for alpha in alpha_values for name, model in classifiers]

# Parallelize the computation across available cores
results = Parallel(n_jobs=7)(
    delayed(compute_best_result)(alpha, name, model, X, Y) for alpha, name, model, X, Y in tasks
)

# Aggregate results to find the best for each classifier
for classifier_name, auc, features, alpha in results:
    current_best_auc, _, _ = best_results_by_classifier[classifier_name]
    if auc > current_best_auc:
        best_results_by_classifier[classifier_name] = (auc, features, alpha)

# The final results are stored in the 'best_results_by_classifier' dictionary.
best_results_by_classifier

{'Logistic Default': (0.7542617385940794,
  [224,
   3,
   204,
   138,
   88,
   21,
   183,
   22,
   227,
   142,
   24,
   244,
   13,
   103,
   119,
   184,
   86,
   26,
   38,
   155,
   229,
   222,
   122,
   253,
   126,
   174,
   25,
   82,
   77,
   137,
   157,
   182,
   42,
   216,
   49,
   232,
   115,
   133,
   196,
   231,
   217,
   90,
   210,
   116,
   202,
   59,
   156,
   109,
   65,
   20,
   52,
   201,
   113,
   68,
   243,
   214,
   206,
   185,
   57,
   197,
   78,
   64,
   1,
   221,
   69,
   80,
   91,
   11,
   176,
   29,
   35,
   123,
   117,
   33,
   67,
   147,
   215,
   61,
   154,
   207,
   51,
   252,
   62,
   190,
   179,
   30,
   124,
   167,
   148,
   28,
   203,
   168,
   53,
   146,
   5],
  0.3792690190732238)}